In [2]:
import base64
from dotenv import load_dotenv
from anthropic import Anthropic
from anthropic.types import Message

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [3]:
def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)

def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)

def chat( messages, system=None, temperature=1.0, stop_sequences=[], tools=None, thinking=False, thinking_budget=1024 ):

    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools: params["tools"] = tools
    if system: params["system"] = system
    if thinking: params["thinking"] = {
                "type": "enabled",
                "budget_tokens": thinking_budget,
    }

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [4]:
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [5]:
with open("images/prop7.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []
add_user_message(messages, [
    # Image block
    {
        "type":"image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes
        }
    },
    # text block
    {
        "type": "text",
        "text": prompt
    }
]
)

chat(messages)

{
  "id": "msg_01Q3pFsyeM6aaedmRZSi4E7M",
  "container": null,
  "content": [
    {
      "citations": null,
      "text": "# Satellite Image Fire Risk Analysis\n\n**1. Residence Identification:**\nThe primary residence is a light-colored (gray/tan) multi-section structure located in the center of the image, featuring an irregular footprint with what appears to be multiple roof sections, surrounded by dense vegetation on all sides with no clearly visible driveway in this view.\n\n**2. Tree Overhang Analysis:**\nMultiple tree canopies directly overhang the residence roof, with dense foliage coverage estimated at 50-75% of the total roof area, particularly concentrated on the northern, eastern, and western portions of the structure where dark shadows from tree canopies are clearly visible on the roofing material.\n\n**3. Fire Risk Assessment:**\nThe overhanging trees create significant wildfire vulnerability through multiple ember catch points across the roof surface, continuous fuel pat

### PDF support


In [6]:
with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []
add_user_message(messages, [
    # Image block
    {
        "type":"document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes
        }
    },
    # text block
    {
        "type": "text",
        "text": "Summarize the document in a 2-3 sentences."
    }
]
)

chat(messages)

{
  "id": "msg_01A2zL7DsaSh7WLm6iAssqKw",
  "container": null,
  "content": [
    {
      "citations": null,
      "text": "# Summary\n\nThis Wikipedia article provides comprehensive information about Earth, the third planet from the Sun and the only known astronomical object harboring life. It covers Earth's physical characteristics (including its 40,000 km circumference, 70.8% ocean coverage, and mean temperature of 14.76\u00b0C), orbital properties, geological formation approximately 4.5 billion years ago, and its dynamic atmosphere composed primarily of nitrogen and oxygen that sustains life and protects the surface from harmful radiation.",
      "type": "text"
    }
  ],
  "model": "claude-sonnet-4-5-20250929",
  "role": "assistant",
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_in

### Citations

In [8]:
with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []
add_user_message(messages, [
    # Image block
    {
        "type":"document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes
        },
        "title": "earth.pdf",
        "citations": {"enabled": True}
    },
    # text block
    {
        "type": "text",
        "text": "How are Earth's atmosphere and oceans were formed?"
    }
]
)

chat(messages)

{
  "id": "msg_019cwa5C77BE7Wd4J8SqnR7h",
  "container": null,
  "content": [
    {
      "citations": [
        {
          "cited_text": "[42]\r\nEarth's atmosphere and oceans were formed by volcanic activity and outgassing.\r\n",
          "document_index": 0,
          "document_title": "earth.pdf",
          "end_page_number": 5,
          "file_id": null,
          "start_page_number": 4,
          "type": "page_location"
        }
      ],
      "text": "Earth's atmosphere and oceans were formed by volcanic activity and outgassing.",
      "type": "text"
    },
    {
      "citations": null,
      "text": " ",
      "type": "text"
    },
    {
      "citations": [
        {
          "cited_text": "[43] Water vapor from\r\nthese sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,\r\nand comets.\r\n",
          "document_index": 0,
          "document_title": "earth.pdf",
          "end_page_number": 5,
          "file_id": null,
          